In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.2,
)

In [3]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated


@tool
def conversion_rate(base_currency:str, target_currency:str) -> float:
    """
    This function fetches the currency conversion factor between a given base currency and a target currency
    """
    link= f"https://v6.exchangerate-api.com/v6/967ab55646c462eacda33415/pair/{base_currency}/{target_currency}"

    response = requests.get(link)
    return response.json()["conversion_rate"]

@tool
def convert(base_currency_value:int, conversion_factor: Annotated[float, InjectedToolArg])-> float:
    '''
    given a currency conversion rate this function calculates the target currency value from a given base currency value
    '''
    return base_currency_value * conversion_factor

In [ ]:
llm_with_tools = model.bind_tools([conversion_rate, convert])
# llm_with_tools = model.bind_tools([convert])

In [5]:
messages = [HumanMessage('What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr')]

In [6]:
ai_message = llm_with_tools.invoke(messages)
print(ai_message)

c:\Users\kumar\OneDrive\Desktop\Resume\LangChain\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3610: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


content=[] additional_kwargs={'function_call': {'name': 'conversion_rate', 'arguments': '{"target_currency": "INR", "base_currency": "USD"}'}, '__gemini_function_call_thought_signatures__': {'call_3070540720664187292': 'EmAKXgFpFH0TsbseBm7W1/r0GZ3Ya02wdRnfS8i0HFNnmPnmr9wmW3kvWjaPYeq5tVROadcFr8Ojz/cgI86qtSLc7jqBBN+E2BX9+eYncZsTCn3L/tc1QnnxsejmBG44eEk='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a1d015-fa67-7620-ac37-0e1d72f0cf25-0' tool_calls=[{'name': 'conversion_rate', 'args': {'target_currency': 'INR', 'base_currency': 'USD'}, 'id': 'call_3070540720664187292', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 159, 'output_tokens': 26, 'total_tokens': 185, 'input_token_details': {'cache_read': 0}}


In [8]:
ai_message.tool_calls 

[{'name': 'conversion_rate',
  'args': {'target_currency': 'INR', 'base_currency': 'USD'},
  'id': 'call_3070540720664187292',
  'type': 'tool_call'}]

In [7]:
messages.append(ai_message)

In [9]:
conversion_factor = conversion_rate.invoke(
    ai_message.tool_calls[0]["args"]
)
print(conversion_factor)

messages.append(f'conversion_factor: {conversion_factor}')


94.5405


In [10]:
# messages.append(
#     ToolMessage(
#         content=str(conversion_factor),
#         tool_call_id=ai_message.tool_calls[0]["id"]
#     )
# )

In [11]:
print(messages)

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={}), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'conversion_rate', 'arguments': '{"base_currency": "USD", "target_currency": "INR"}'}, '__gemini_function_call_thought_signatures__': {'call_2207024': 'El4KXAERTTIPOCO6PJMoflkPL3tpB7IP9X9tiVQ7K26VeSWbhjupulxK6AUrimg7mpOxl3ha3331W812zxbTXEvTbIb7hxJcs+EVyINXRo5X1BqGGxq0Z2JNaSTb2+5j'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a06aa6-ec13-7322-b98a-e5a9b1d9a7ac-0', tool_calls=[{'name': 'conversion_rate', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_2207024', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 159, 'output_tokens': 26, 'total_tokens': 185, 'input_token_details': {'cache_read': 0

In [12]:
llm_with_tools = model.bind_tools([convert])

query = convert.invoke({
    "base_currency_value": 10,
    "conversion_factor": 0.01058
})

In [13]:
# ai_message.tool_calls[0]['args']['conversion_factor'] = conversion_factor

# tool_message2 = convert.invoke(ai_message.tool_calls[0])

# messages.append(tool_message2)

In [14]:
messages.append(f'converted value: {query}')

In [15]:
# print(model.invoke(messages).content)

In [14]:
for tool_call in ai_message.tool_calls:

    if tool_call['name'] == 'conversion_rate':

        tool_message1 = conversion_rate.invoke(tool_call)

        # conversion_rate already returns a float
        conversion_rate1 = float(tool_message1.content)

        messages.append(tool_message1)

    if tool_call['name'] == 'convert':

        tool_call['args']['conversion_factor'] = conversion_rate1

        tool_message2 = convert.invoke(tool_call)

        messages.append(tool_message2)

In [17]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'conversion_rate', 'arguments': '{"target_currency": "INR", "base_currency": "USD"}'}, '__gemini_function_call_thought_signatures__': {'call_3070540720664187292': 'EmAKXgFpFH0TsbseBm7W1/r0GZ3Ya02wdRnfS8i0HFNnmPnmr9wmW3kvWjaPYeq5tVROadcFr8Ojz/cgI86qtSLc7jqBBN+E2BX9+eYncZsTCn3L/tc1QnnxsejmBG44eEk='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1d015-fa67-7620-ac37-0e1d72f0cf25-0', tool_calls=[{'name': 'conversion_rate', 'args': {'target_currency': 'INR', 'base_currency': 'USD'}, 'id': 'call_3070540720664187292', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 159, 'output_tokens': 26, 'total_tokens': 185, 'input_tok

In [19]:
llm_with_tools.invoke(messages).content
print(llm_with_tools.invoke(messages).content)

c:\Users\kumar\OneDrive\Desktop\Resume\LangChain\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3610: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\kumar\OneDrive\Desktop\Resume\LangChain\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3610: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[]
